# Lab 3: Plan It, Then Check It


## How this Colab lab works

Use one shared team copy. One person is the **driver** and runs code; the other is the **statistical navigator** and checks the unit, denominator, split, and claim. Switch roles at the marked handoff.

1. Record the individual prediction before revealing output.
2. Run the instructor example.
3. Modify the supplied code with your partner.
4. Run the supplied structural check.
5. Inspect actual images or text when requested.
6. Write the independent handoff in your own words.

The notebook file is shared; each collaborator's temporary Colab runtime is not. Avoid two people executing different versions simultaneously. Save the notebook before switching drivers.


In [ ]:
# Standard Colab setup - run once
from pathlib import Path
import hashlib
import json
import random
import sys
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 2027
random.seed(SEED)
np.random.seed(SEED)

COURSE_REPO_RAW_URL = 'https://raw.githubusercontent.com/skgallagher/stat-methods-ai-public/main'
COURSE_DATA_BASE_URL = COURSE_REPO_RAW_URL + '/data/course'
COURSE_DATA_GROUPS = ['cfpb']
RUNTIME_ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
DATA_ROOT = RUNTIME_ROOT / 'stat_ai_data'

# A local notebook may start in the repository root or a nested week folder.
# Search upward for frozen course data before falling back to a download.
search_roots = [Path.cwd(), *Path.cwd().parents]
LOCAL_RELEASE = next((
    root / 'data' / 'course'
    for root in search_roots
    if (root / 'data' / 'course' / 'manifest.json').exists()
), None)
LOCAL_SMOKE = next((
    root / 'data' / 'smoke'
    for root in search_roots
    if (root / 'data' / 'smoke').exists()
), None)
online_release = False
if LOCAL_RELEASE is not None:
    DATA_ROOT = LOCAL_RELEASE
    data_source = 'local frozen release'
elif LOCAL_SMOKE is not None:
    DATA_ROOT = LOCAL_SMOKE
    data_source = 'local synthetic smoke fixture (development only)'
elif (DATA_ROOT / 'manifest.json').exists():
    cached_manifest = json.loads((DATA_ROOT / 'manifest.json').read_text())
    if 'smoke fixture' in cached_manifest.get('bundle_type', ''):
        data_source = 'existing local synthetic smoke fixture (development only)'
    else:
        cached_files = cached_manifest.get('files', [])
        requested_files = [
            item for item in cached_files
            if Path(item['path']).parts[0] in COURSE_DATA_GROUPS
        ]
        def cached_sha256(path):
            digest = hashlib.sha256()
            with path.open('rb') as stream:
                for chunk in iter(lambda: stream.read(1024 * 1024), b''):
                    digest.update(chunk)
            return digest.hexdigest()
        cache_complete = (
            cached_manifest.get('release_status') == 'student_release'
            and requested_files
            and all(
                (DATA_ROOT / item['path']).exists()
                and cached_sha256(DATA_ROOT / item['path']) == item['sha256']
                for item in requested_files
            )
        )
        if cache_complete:
            data_source = 'existing verified runtime cache'
        else:
            online_release = True
else:
    online_release = True

if online_release:
    # Build the package directory as path components so notebook/Markdown
    # converters cannot turn the underscore into a literal escaped path.
    helper_package = 'course' + '_helpers'
    helper_target = RUNTIME_ROOT / helper_package / '__init__.py'
    helper_target.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(
        COURSE_REPO_RAW_URL + '/course_helpers/__init__.py', helper_target
    )
    runtime_import_root = str(RUNTIME_ROOT)
    if runtime_import_root not in sys.path:
        sys.path.insert(0, runtime_import_root)
    from course_helpers import ensure_course_data
    DATA_ROOT = ensure_course_data(
        COURSE_DATA_BASE_URL,
        DATA_ROOT,
        groups=COURSE_DATA_GROUPS,
    )
    data_source = 'public GitHub student release'

print('Setup complete. Data root:', DATA_ROOT)
print('Data source:', data_source)
print('Requested groups:', COURSE_DATA_GROUPS)


**Today:** We will read through one complete evaluation plan, find two problems
in a weak plan, and run a small practice analysis. We will use the CFPB
complaint data again in Homework 3.

In this lab, we use 2022 for training and 2023 to practice choosing a model.
We leave 2024 and 2025 untouched because you will use them in Homework 3.

## Read through a complete plan — 10 minutes

Imagine a tool that reads a public complaint narrative and recommends one of
five CFPB product categories. An analyst reviews the recommendation before
making the final choice.

Here are the decisions behind the plan:

1. **What is the model trying to predict?** For each complaint, the model
   predicts the product category recorded by the CFPB. That category is the
   decision stored in the data. It does not prove that every complaint has one
   objectively correct category.
2. **Who does the result describe?** The teaching file contains public
   narratives from five product categories. It does not represent complaints
   without public narratives, and the five categories are equally common here
   even though they may not be equally common outside this file.
3. **What comparisons do we need?** First, logistic regression must do better
   than a rule that ignores the complaint text. Then a transformer must do
   enough better than logistic regression to justify trying it in a pilot.
4. **Why do we need three data sets?** We use the training data to fit each
   model. We use the validation data to choose settings. We use the test data
   once, after every choice has been made, to estimate performance on new data.
5. **How will we compare the methods?** Every method predicts the same
   complaints. We compare their predictions complaint by complaint and report
   an interval for each difference in accuracy. Before seeing the test results,
   we also decide how much better the transformer must be to justify a pilot.
6. **What decision can this study support?** At most, the results could support
   a pilot in which a person reviews each recommendation. They could not
   justify automatic routing, prove that the recorded category is correct, or
   show that better routing improves complaint outcomes.

Now make three choices of your own. Different answers can work; give one short
reason for each.

| Your choice | Answer and reason |
|---|---|
| Accuracy or average recall across the five products? | TODO |
| What minimum transformer improvement would justify a pilot? | TODO |
| Can versions of the same complaint appear in both sets? | TODO |

## Find two problems in a weak plan — 6 minutes

An AI assistant received only “determine whether the transformer is better” and
proposed:

> Randomly split all years, remove short narratives and rare words using the
> complete dataset, try several category groupings, and report whichever
> result is best. Deploy the transformer if it wins.

With a partner, pick the two parts of this plan that worry you most. For each
one, say what could go wrong and how you would change it.

| Part of the plan | What could go wrong? | How would you change it? |
|---|---|---|
| TODO | TODO | TODO |
| TODO | TODO | TODO |

What important problem would remain even after those changes? TODO

## Check two ways to split the data — 10 minutes

The code below uses only the 2022 and 2023 rows. Complaints in the same
`similarity_group` have the same or very similar text. Before running the code,
predict which split will put a similarity group in both sets.

In [ ]:
from sklearn.model_selection import train_test_split

cfpb_root = DATA_ROOT / "cfpb"
complaints = pd.read_csv(cfpb_root / "complaints.csv")
complaints["date_received"] = pd.to_datetime(complaints["date_received"])
assert complaints["complaint_id"].is_unique
assert len(complaints) == 400

development = complaints.query("received_year in [2022, 2023]").copy()
time_train = development.query("received_year == 2022").copy()
time_validation = development.query("received_year == 2023").copy()

row_train, row_validation = train_test_split(
    development,
    test_size=0.50,
    random_state=SEED,
    stratify=development["product"],
)

def shared_similarity_groups(left, right):
    return set(left["similarity_group"]) & set(right["similarity_group"])

development_counts = pd.crosstab(
    development["received_year"], development["product"]
)
split_check = pd.DataFrame([
    {
        "split": "2022 train / 2023 practice validation",
        "training_rows": len(time_train),
        "validation_rows": len(time_validation),
        "shared_similarity_groups": len(shared_similarity_groups(
            time_train, time_validation
        )),
    },
    {
        "split": "random rows from 2022-2023",
        "training_rows": len(row_train),
        "validation_rows": len(row_validation),
        "shared_similarity_groups": len(shared_similarity_groups(
            row_train, row_validation
        )),
    },
])

display(development_counts)
display(split_check)

# TODO: add an assertion that both time-based sets contain all five products.
# TODO: add an assertion that the time-based sets contain 100 rows each.

Fill in only the results you need:

| Result | Value |
|---|---:|
| Complaints for each product in each year | TODO |
| Shared groups in the time split | TODO |
| Shared groups in the random-row split | TODO |

If the intended use is complaints received **after 2023**, which split is the
better practice test? What did the duplicate check show? TODO

## Fit one practice model — 10 minutes

We will use 2022 to fit three versions of logistic regression and 2023 to
choose among them. Homework 3 makes a new choice using 2024, so the numbers
below are not homework answers.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline

PRACTICE_C_VALUES = [0.25, 1.0, 4.0]

def make_practice_model(C):
    return make_pipeline(
        TfidfVectorizer(min_df=2, ngram_range=(1, 2)),
        LogisticRegression(C=C, max_iter=2000, random_state=SEED),
    )

practice_rows = []
for C in PRACTICE_C_VALUES:
    model = make_practice_model(C)
    model.fit(time_train["narrative"], time_train["product"])
    prediction = model.predict(time_validation["narrative"])
    practice_rows.append({
        "C": C,
        "2023_accuracy": accuracy_score(time_validation["product"], prediction),
    })

practice_results = pd.DataFrame(practice_rows)
selected_C = float(
    practice_results.sort_values(
        ["2023_accuracy", "C"], ascending=[False, True]
    ).iloc[0]["C"]
)
baseline_accuracy = time_validation["product"].value_counts(
    normalize=True
).max()

display(practice_results)
print("Selected C:", selected_C)
print("Baseline accuracy:", baseline_accuracy)

# TODO: add an assertion that selected_C came from PRACTICE_C_VALUES.

| Result | Value |
|---|---:|
| Baseline accuracy from always predicting the most common product | TODO |
| Selected `C` and its accuracy | TODO |

Complete two sentences:

- On the 2023 practice rows, the selected model ________.  
- TF-IDF belongs inside the pipeline because ________.

## Before you leave — 7 minutes

### Independent handoff response

Complete this individually.

1. In one sentence, say what the simple model and the transformer each need to
   beat. TODO
2. What problem did the split or pipeline code help with? TODO
3. What problem with public narratives is still there? TODO
4. When the case changes to GitHub issues in Homework 3, which two choices must
   you make again? TODO

## Exit ticket — 2 minutes

What result or decision did we intentionally leave for Homework 3?

**For Homework 3:** In Problem 1, you make similar choices for GitHub issues.
In Problem 2, you train with 2022–2023, choose a model with 2024, and test it
once with 2025. Nothing in this lab reveals those 2024 or 2025 results.

### Exit-ticket response

TODO


## Before leaving

- [ ] The notebook runs in order through the required handoff.
- [ ] Denominators, units, and evaluated population are visible.
- [ ] Required image/text cases are displayed rather than merely described.
- [ ] The driver and navigator switched at least once.
- [ ] Each person wrote the independent handoff.
- [ ] The named artifact was saved for the homework or project.
